# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AmanDbz1101/FlyRank-/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Ranking** (implemented through scoring / estimated probability).

The question is not "will this page decline?" — that is a yes/no classification. The question is **"which declining pages should an editor look at first?"** That is a ranking problem: given a fixed budget of editor attention per week, we need to order pages so that the most impactful reviews happen first.

We will train a binary classifier (declining vs not-declining) and use its predicted probability as a **score** to rank pages. The probability itself is a means to the rank — the real output is the ordered list.

In [ ]:
# Confirm: ranking via scoring
# This cell will show that we have a binary label and we will rank by predicted probability.
import pandas as pd
import numpy as np
print("Task type: Ranking (via binary classification score)")

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target:** `trend_direction == "down"` — a page whose search impressions dropped more than 20% in the last 30 days compared to the prior 30 days.

**Where it comes from:** An observed outcome. The label is derived from `impressions_last_30d` vs `impressions_prev_30d`, which are measured (not guessed). The trend is computed, not hand-labeled. The prep script encodes this as `is_declining_label`.

Because `trend_direction` and `trend_pct` are computed from impressions windows, they **must never be features** — they are the label source. We will use only the features that predate or are independent of those windows.

**Proxy note:** "Declining" is itself a proxy for the business concern (a page losing relevance/rankings). We accept this because the observed drop in impressions is the closest measurable signal we have before editor attention is justified.

In [ ]:
# Load data and show target distribution
import os
# Find the repo root by walking up from cwd
REPO_ROOT = os.getcwd()
while not os.path.exists(os.path.join(REPO_ROOT, '.git')) and os.path.dirname(REPO_ROOT) != REPO_ROOT:
    REPO_ROOT = os.path.dirname(REPO_ROOT)
DATA_PATH = os.path.join(REPO_ROOT, 'data', 'raw', 'content_refresh_anonymized.csv')
df = pd.read_csv(DATA_PATH)
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

print('Target distribution:')
print(df['is_declining_label'].value_counts())
print(f"\nDecline rate: {df['is_declining_label'].mean():.1%}")
print(f"Rows: {len(df)}")

## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Primary: Precision@K** (where K = editor capacity, e.g. 50, 100, or 200 pages per week).

We rank all pages by predicted probability of decline, take the top K, and measure what fraction of those are actually declining. This directly answers the business question: "of the pages we review, how many were worth reviewing?"

**Secondary: Average Precision (AP), ROC AUC.** AP summarises precision across all rank cutoffs; ROC AUC tells us how well the model separates the two classes overall.

**Why this metric:** Editor time is the scarce resource. A false positive (reviewing a healthy page) costs ~10 minutes of editor time. A false negative (missing a declining page) means the page keeps losing traffic until the next review cycle. Precision@K aligns the metric with the operational constraint.

In [ ]:
# Show current decline rate as the "no model" baseline for Precision@K
baseline_decline_rate = df['is_declining_label'].mean()
print(f'Random Precision@K (any K): {baseline_decline_rate:.1%}')
print('A model must beat this at the top of the ranked list to be useful.')
print(f'\nTotal declining pages: {df["is_declining_label"].sum():,}')
print(f'Total non-declining:   {(1 - df["is_declining_label"]).sum():,}')

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**One row = one content page** (article, guide, comparison page, etc.) — identified by `content_id`. Each row carries:
- The page's observable features (keyword context, content properties, recent traffic totals)
- Its performance trajectory (encoded as the target label)

We load the 30,000-row slice and show a representative subset of feature columns alongside the target.

In [ ]:
# Show the unit of analysis: one row = one content page
FEATURE_DEMO = [
    'content_id', 'client_id', 'content_type', 'main_intent',
    'search_volume', 'competition', 'word_count', 'content_age_days',
    'days_since_last_update', 'impressions_90d', 'clicks_90d',
    'ctr', 'avg_position', 'engagement_rate', 'ai_traffic_pct',
    'is_declining_label'
]

display_df = df[FEATURE_DEMO].copy()
print(f'Shape: {display_df.shape}')
print(f'One row = one content page  |  {display_df["content_id"].nunique():,} unique pages')
print(f'\nFirst 5 rows (showing unit of analysis):')
display_df.head()

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule — e.g. "flag any page where `ctr < 0.5` AND `avg_position > 10`" — is brittle for three reasons:

1. **Interaction effects.** Decline is not driven by one or two signals. A page with low CTR but strong engagement (high scroll_rate, many returning users) may recover; a page with good CTR but declining AI-referred traffic (ai_traffic_pct dropping) may be at risk. Fixed rules miss these interactions.

2. **Signal strength varies by client and content type.** What counts as "low" CTR for a `feedly article` differs from a `keyword article`. A fixed threshold at the dataset level would be wrong for many subgroups; ML can learn per-group patterns from the data.

3. **Noise and missingness.** `scroll_rate` can exceed 100, `avg_position = 0` means "no data", keyword columns are missing for entire content types. ML models (especially tree-based ones) handle missing values, non-linear relationships, and noisy inputs more gracefully than hand-tuned thresholds.

In short: the boundary between declining and stable is a fuzzy, high-dimensional manifold — exactly where a learned model outperforms a human-written rule.

In [ ]:
# Demonstrate why fixed rules are hard: show correlations (or lack thereof)
signal_cols = ['ctr', 'avg_position', 'engagement_rate', 'scroll_rate',
               'ai_traffic_pct', 'search_volume', 'content_age_days',
               'days_since_last_update', 'impressions_90d']

corr = df[signal_cols + ['is_declining_label']].corr()['is_declining_label'].drop('is_declining_label')
print('Correlation of each signal with the decline target:')
for col, val in corr.sort_values().items():
    print(f'  {col:30s}  {val:+.3f}')
print()
print('No single signal strongly correlates with decline.')
print('The relationship is multivariate — no fixed rule on one or two signals will work well.')

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.